In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import (
    col, expr, to_date, when, lower, lit
)

In [0]:
@dp.temporary_view()
def transformed_ecomm_transactions():
    # Read bronze transactions as streaming
    df = spark.readStream.table(
        "electroniz_catalog.electroniz_bronze_schema.electroniz_bronze_ecomm_transactions"
    )

    # Add sequential ID
    df = df.withColumn("ecomm_customer_id", expr("uuid()"))

    # Extract fields from MAP column
    df = df.select(
        col("ecomm_customer_id"),
        col("data").getItem("customer_name").alias("customer_name"),
        col("data").getItem("address").alias("address"),
        col("data").getItem("city").alias("city"),
        col("data").getItem("postalcode").alias("postalcode"),
        col("data").getItem("country").alias("country"),
        col("data").getItem("phone").alias("phone"),
        col("data").getItem("email").alias("email"),
        col("data").getItem("product_name").alias("product_name"),
        col("data").getItem("order_date").alias("order_date"),
        col("data").getItem("currency").alias("currency"),
        col("data").getItem("order_mode").alias("order_mode"),
        col("data").getItem("sale_price").alias("sale_price"),
        col("data").getItem("order_number").alias("order_number"),
        col("_ingested_at")
    )

    # Cast sale_price to double
    df = df.withColumn("sale_price", col("sale_price").cast("double"))

    # Cast order_number to integer
    df = df.withColumn("order_number", col("order_number").cast("integer"))

    # Cast order_date (DD/MM/YYYY) to date type
    df = df.withColumn("order_date", to_date(col("order_date"), "dd/MM/yyyy"))

    # Standardize country abbreviations
    df = df.withColumn("country",
        when(col("country").isin("United States", "US", "U.S.", "America"), lit("US"))
        .when(col("country").isin("United Kingdom", "U.K"), lit("UK"))
        .when(col("country").isin("India", "IN"), lit("IND"))
        .when(col("country").isin("Canada", "CA"), lit("CAN"))
        .otherwise(col("country"))
    )

    # Read products dimension table as static
    products = spark.read.table(
        "electroniz_catalog.electroniz_bronze_schema.electroniz_bronze_products"
    )

    # Save reference before join to avoid ambiguity
    txn_df = df

    # INNER JOIN on LOWER(product_name) for case-insensitive matching
    df = txn_df.join(
        products,
        lower(txn_df["product_name"]) == lower(products["product_name"]),
        "inner"
    )

    # Select all columns using txn_df["column"] syntax to avoid ambiguity
    df = df.select(
        txn_df["ecomm_customer_id"],
        txn_df["customer_name"],
        txn_df["address"],
        txn_df["city"],
        txn_df["postalcode"],
        txn_df["country"],
        txn_df["phone"],
        txn_df["email"],
        txn_df["product_name"],
        txn_df["order_date"],
        txn_df["currency"],
        txn_df["order_mode"],
        txn_df["sale_price"],
        txn_df["order_number"],
        products["product_code"],
        products["product_category"],
        txn_df["_ingested_at"]
    )

    return df

In [0]:
dp.create_streaming_table(
    name="electroniz_catalog.electroniz_silver_schema.electroniz_silver_ecomm_transactions",
    comment="Silver ecommerce transactions with quality rules",
    table_properties={"quality": "silver"},
    expect_all_or_drop={
        "valid_customer_name": "customer_name IS NOT NULL",
        "valid_address": "address IS NOT NULL",
        "valid_email_not_null": "email IS NOT NULL",
        "valid_order_number": "order_number IS NOT NULL",
        "valid_email_format": "email RLIKE '^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\\\.[a-zA-Z]{2,}$'",
        "valid_currency": "currency IN ('USD', 'INR', 'CAD', 'EUR')"
    }
)


dp.create_auto_cdc_flow(
    target="electroniz_catalog.electroniz_silver_schema.electroniz_silver_ecomm_transactions",
    source="transformed_ecomm_transactions",
    keys=["order_number"],
    sequence_by="_ingested_at",
    stored_as_scd_type=1
)